# पाठ १३ - कोग्नी नॉलेज ग्राफ्ससहित एजेन्ट मेमोरी


## सेटअप

यो नोटबुकले [**Cognee**](https://www.cognee.ai/) ज्ञान ग्राफहरू र **Microsoft Agent Framework** (MAF) प्रयोग गरी स्थायी स्मृति सहितको एक बुद्धिमान **कोडिङ सहायक** कसरी निर्माण गर्ने देखाउँछ।

Cognee अव्यवस्थित पाठलाई संरचित, क्वेरी गर्न मिल्ने ज्ञान ग्राफमा परिणत गर्दछ जुन भेक्टर एम्बेडिङहरूले समर्थित हुन्छ — तपाईंको एजेन्टलाई धनी, सम्बन्ध-सचेत लामो समयसम्मको स्मृति प्रदान गर्दै।

### तपाईंले के सिक्नुहुनेछ
1. **ज्ञान ग्राफ निर्माण गर्नुहोस्**: विकासकर्ता प्रोफाइलहरू र उत्तम अभ्यासहरूलाई संरचित, क्वेरी गर्न मिल्ने ज्ञानमा परिणत गर्नुहोस्।
2. **Cognee लाई MAF सँग एकीकृत गर्नुहोस्**: `@tool` कार्यहरू प्रयोग गरी MAF एजेन्टलाई Cognee को ज्ञान ग्राफ क्वेरी गर्न दिनुहोस्।
3. **सत्र-सचेत संवादहरू**: एउटै सत्रमा धेरै प्रश्नहरूमा सन्दर्भ कायम राख्नुहोस्।
4. **लामो समयको स्मृति**: महत्त्वपूर्ण ज्ञानलाई सत्रहरूमा कायम राख्नुहोस् र नयाँ संवादहरूमा पुनः प्राप्त गर्नुहोस्।

### पूर्वआवश्यकताहरू
- Python 3.9+
- क्षेत्रीय रूपमा Redis चलिरहेको (`docker run -d -p 6379:6379 redis`) सत्र व्यवस्थापनको लागि
- एउटा LLM API कुञ्जी (जस्तै OpenAI) — `.env` मा `LLM_API_KEY` सेट गरिएको हुनुहोस्
- `.env` मा `CACHING=true` (Cognee सत्रहरूका लागि आवश्यक)
- एउटा Azure AI Foundry परियोजना जसमा एक तैनात गरिएको च्याट मोडेल छ
- `.env` मा `AZURE_AI_PROJECT_ENDPOINT` र `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI प्रमाणित गरिएको (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## एजेन्ट मेमोरीका प्रकारहरू

यो नोटबुकले मुख्य पाठ १३ नोटबुकका तीनै प्रकारका मेमोरीहरूलाई अन्वेषण गर्छ, तर लामो अवधि मेमोरी ब्याकएन्डको रूपमा Cognee प्रयोग गर्छ:

| मेमोरी प्रकार | संयन्त्र | आयु अवधि |
|---|---|---|
| **काम गर्ने** | `agent.create_session()` (MAF) | एकल संवाद |
| **छोटो अवधिको** | Cognee सत्र क्यास (Redis) | एकल सत्र |
| **लामो अवधिको** | Cognee ज्ञान ग्राफ + भेक्टरहरू | स्थायी |

### Cognee को मेमोरी वास्तुकला
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## कोग्नी स्टोरेज तयार गर्नुहोस्


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## भाग १ — ज्ञान आधार निर्माण

हामी हाम्रो कोडिङ सहायकको लागि समग्र ज्ञान आधार बनाउन तीन प्रकारका डाटा सङ्कलन गर्छौं:

1. **विकासकर्ता प्रोफाइल** — व्यक्तिगत विशेषज्ञता र प्राविधिक पृष्ठभूमि  
2. **पाइथन उत्कृष्ट अभ्यासहरू** — व्यावहारिक मार्गनिर्देशहरूसहित पाइथनको ज़ेन  
3. **ऐतिहासिक संवादहरू** — विकासकर्ता र AI सहायकहरू बीचका विगतका प्रश्नोत्तर सत्रहरू


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## ज्ञान ग्राफलाई दृश्यरूप दिनुहोस्

Cognee ले यसले निकालिएको ईकाइहरू र सम्बन्धहरूको अन्तरक्रियात्मक HTML दृश्य प्रस्तुत गर्न सक्छ।


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## मेमोरीलाई मेमिफाइले समृद्ध बनाउनुहोस्

`memify()` ज्ञान ग्राफ विश्लेषण गर्दछ र बौद्धिक नियमहरू सिर्जना गर्दछ — ढाँचाहरू, उत्तम अभ्यासहरू, र अवधारणाहरू बीचको सम्बन्धहरू पहिचान गर्दै।


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## भाग २ — Cognee उपकरणहरूसँग MAF एजेन्ट

अब हामी एक MAF एजेन्ट सिर्जना गर्छौं जसले `@tool` फङ्सनहरू मार्फत Cognee को ज्ञान ग्राफमा क्वेरी गर्न सक्छ। यसले एजेन्टलाई सत्रहरूको माध्यमबाट संवादात्मक सन्दर्भ कायम राख्दै ग्राफ-जानकारी सेमान्टिक खोजको पूर्ण शक्ति उपयोग गर्न दिन्छ।


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## सेसन्ससँग काम गर्ने स्मृति

`AgentSession` (जुन `agent.create_session()` मार्फत सिर्जना गरिएको हो) सेसन भित्र काम गर्ने स्मृति प्रदान गर्दछ। एजेन्टले पहिलेका सन्देशहरूमा फिर्ता हेर्न सक्छ भने साथै Cognee को दीर्घकालीन ज्ञान ग्राफसँग सोधपुछ गर्न सक्छ।


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## नयाँ सत्र — दीर्घकालीन स्मृति कायम रहन्छ

नयाँ सत्र सुरू गर्दा कार्य स्मृति सफा हुन्छ, तर Cognee ज्ञान ग्राफ अझै उपलब्ध छ। एजेन्टले बिल्कुल नयाँ संवादमा त्यही दीर्घकालीन ज्ञान पुनः प्राप्त गर्न सक्छ।


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## संक्षेप

यस नोटबुकमा तपाईंले एउटा कोडिङ सहायकरूपमा निर्माण गर्नुभयो जसले **MAF को कार्यस्मृति** (`agent.create_session()`) लाई **Cognee को दीर्घकालीन ज्ञान ग्राफ** सँग मिलाउँछ।

### तपाईंले के सिक्नुभयो
1. **ज्ञान ग्राफ निर्माण**: Cognee ले असंरचित पाठहरू समावेश गरी ग्राफ + भेक्टर स्मृति तयार गर्छ।
2. **memify सँग ग्राफ समृद्धि**: तपाईंको विद्यमान ग्राफमा आधारित तथ्यहरू र धनी सम्बन्ध थप्ने।
3. **MAF + Cognee एकीकरण**: `@tool` फंक्शन्सले MAF एजेन्टलाई Cognee को ग्राफ स्वाभाविक रूपमा सोध्न दिन्छ।
4. **कार्यस्मृति + दीर्घकालीन स्मृति**: `AgentSession` (मार्फत `agent.create_session()`) ले सत्र सन्दर्भ दिन्छ भने Cognee स्थायी ज्ञान प्रदान गर्छ।
5. **NodeSets सँग फिल्टर्ड खोज**: ज्ञान ग्राफका विशिष्ट उपसमूहहरू लक्षित गर्ने (जस्तै केवल सिद्धान्तहरू)।

### मुख्य बुँदाहरू
- **Cognee** कच्चा पाठलाई संरचित, सम्बन्ध-समझदार स्मृतिमा परिवर्तन गर्छ — फ्ल्याट भेक्टर स्टोरभन्दा बढी शक्तिशाली।
- **`@tool` फंक्शन्स** ले MAF एजेन्टहरू र बाह्य ज्ञान प्रणालीहरूलाई सफा रूपमा जोड्छ।
- **`AgentSession`** (मार्फत `agent.create_session()`) ले वार्तालाप-प्रति सन्दर्भ दीर्घकालीन ज्ञानबाट अलग राख्छ।
- समान ज्ञान ग्राफ धेरै सत्र र एजेन्टहरूको सेवा गर्दछ।

### वास्तविक संसारका अनुप्रयोगहरू
- **डेभलपर कोपाइलटहरू**: कोड समीक्षा, घटना विश्लेषण, आर्किटेक्चर सहायकहरू
- **ग्राहकमुखी कोपाइलटहरू**: उत्पादन दस्तावेज, FAQ, र CRM नोट्समाथि समर्थन एजेन्टहरू
- **आन्तरिक विशेषज्ञ कोपाइलटहरू**: नीतिहरू, कानूनी वा सुरक्षा सहायकहरू निर्देशनहरू अनुसार तर्क गर्ने
- **एकीकृत डेटा तहहरू**: संरचित र असंरचित डेटालाई एउटै सोध्न मिल्ने ग्राफमा जोड्ने

### अर्को चरणहरू
- Cognee मा कालगत जागरूकता परीक्षण गर्ने
- डोमेन-विशेष ग्राफ गुणस्तरको लागि OWL ओन्टोलोजी परिभाषित गर्ने
- समयक्रमसँग सुधार गर्न प्रयोगकर्ताको प्रतिक्रिया लूपहरू थप्ने
- समान Cognee स्मृति तह साझेदारी गर्ने बहु-एजेन्ट प्रणालीहरूमा विस्तार गर्ने


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकृति**:
यो दस्तावेज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी शुद्धतालाई ध्यान दिने प्रयास गर्छौं, तर कृपया बुझ्नुस् कि स्वचालित अनुवादमा त्रुटि वा अशुद्धि हुन सक्छ। मूल दस्तावेजलाई यसको मूल भाषामा प्रमाणिक स्रोतको रूपमा मानिनु पर्छ। महत्वपूर्ण जानकारीको लागि, व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट हुने कुनै पनि गलत बुझाइ वा व्याख्याप्रति हामी जिम्मेवार हुँदैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
